In [1]:
import pandas as pd
print("Library Pandas berhasil dipanggil! Sistem siap memproses data.")

Library Pandas berhasil dipanggil! Sistem siap memproses data.


In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore') # Mengabaikan pesan peringatan yang tidak penting

print("--- Memulai Proses Pembacaan Data ---")

# 1. Membaca keempat file dataset (Mundur satu folder ke '../data/')
df_cnn = pd.read_excel('../data/dataset_cnn_10k.xlsx')
df_kompas = pd.read_excel('../data/dataset_kompas_4k.xlsx')
df_tempo = pd.read_excel('../data/dataset_tempo_6k.xlsx')
df_hoax = pd.read_excel('../data/dataset_turnbackhoax_10k.xlsx')

print("Berhasil membaca semua file!")

# 2. Memberikan label. Berita dari CNN, Kompas, Tempo adalah 'Valid', Turnbackhoax adalah 'Hoaks'
df_cnn['Label'] = 'Valid'
df_kompas['Label'] = 'Valid'
df_tempo['Label'] = 'Valid'
df_hoax['Label'] = 'Hoaks'

# 3. Menggabungkan semuanya menjadi satu dataset utuh
df_all = pd.concat([df_cnn, df_kompas, df_tempo, df_hoax], ignore_index=True)

print(f"Total baris data yang digabungkan: {len(df_all)} baris")

# 4. Menampilkan 5 data teratas untuk memastikan strukturnya
df_all.head()

--- Memulai Proses Pembacaan Data ---
Berhasil membaca semua file!
Total baris data yang digabungkan: 31726 baris


,Title,Timestamp,FullText,Tags,Author,Url,Label
0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,"Selasa, 21 Feb 2023 21:22 WIB","Jakarta, CNN Indonesia -- Mantan Gubernur DKI ...",anies baswedan;pengajian;pilpres 2024;badan ko...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Valid
1,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,"Selasa, 21 Feb 2023 20:46 WIB","Medan, CNN Indonesia -- Gubernur Sumatera Utar...",edy rahmayadi;pemilu 2024;pilkada 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Valid
2,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,"Selasa, 21 Feb 2023 20:33 WIB","Jakarta, CNN Indonesia -- Partai Kebangkitan B...",ida fauziyah;pkb;pemilu 2024;pileg 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Valid
3,Gede Pasek Doakan AHY Jadi Capres atau Cawapres,"Selasa, 21 Feb 2023 19:58 WIB","Jakarta, CNN Indonesia -- Ketua Umum Partai Ke...",gede pasek suardika;ahy;pilpres 2024;pemilu 20...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Valid
4,PKN Siapkan Jabatan Khusus Buat Anas Urbaningr...,"Selasa, 21 Feb 2023 18:56 WIB","Jakarta, CNN Indonesia -- Dewan Pimpinan Pusat...",anas urbaningrum;pkn;pemilu 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Valid


In [3]:
import re # Library untuk memanipulasi teks (Regular Expression)

print("--- Memulai Proses Preprocessing (Pembersihan Data) ---")

# 1. Memilih kolom yang relevan saja (sesuai rancangan UTS)
df = df_all[['FullText', 'Label']].copy()

# Mengubah nama kolom agar persis seperti di proposal UTS
df.rename(columns={'FullText': 'Teks Berita'}, inplace=True)

# 2. Menangani Missing Values (Data Kosong)
print(f"Jumlah baris sebelum cek data kosong: {len(df)}")
df.dropna(subset=['Teks Berita'], inplace=True) # Menghapus baris yang tidak ada teksnya
print(f"Jumlah baris setelah hapus data kosong: {len(df)}")

# 3. Membuat fungsi NLP Cleaning (Lowercase, hapus tanda baca)
def bersihkan_teks(teks):
    teks = str(teks) # Memastikan formatnya adalah teks (string)
    teks = teks.lower() # Mengubah ke huruf kecil (lowercase)
    teks = re.sub(r'[^a-z\s]', ' ', teks) # Menghapus semua karakter selain huruf (tanda baca, angka, simbol)
    teks = re.sub(r'\s+', ' ', teks).strip() # Merapikan spasi yang berlebihan
    return teks

# 4. Menerapkan fungsi pembersihan ke dalam data kita
print("Sedang membersihkan teks (proses ini mungkin memakan waktu sekitar 1-2 menit)...")
df['Teks Bersih'] = df['Teks Berita'].apply(bersihkan_teks)

print("Pembersihan selesai!")
# Menampilkan 5 data teratas untuk melihat hasilnya
df[['Teks Berita', 'Teks Bersih', 'Label']].head()

--- Memulai Proses Preprocessing (Pembersihan Data) ---
Jumlah baris sebelum cek data kosong: 31726
Jumlah baris setelah hapus data kosong: 31327
Sedang membersihkan teks (proses ini mungkin memakan waktu sekitar 1-2 menit)...
Pembersihan selesai!


,Teks Berita,Teks Bersih,Label
0,"Jakarta, CNN Indonesia -- Mantan Gubernur DKI ...",jakarta cnn indonesia mantan gubernur dki jaka...,Valid
1,"Medan, CNN Indonesia -- Gubernur Sumatera Utar...",medan cnn indonesia gubernur sumatera utara ed...,Valid
2,"Jakarta, CNN Indonesia -- Partai Kebangkitan B...",jakarta cnn indonesia partai kebangkitan bangs...,Valid
3,"Jakarta, CNN Indonesia -- Ketua Umum Partai Ke...",jakarta cnn indonesia ketua umum partai kebang...,Valid
4,"Jakarta, CNN Indonesia -- Dewan Pimpinan Pusat...",jakarta cnn indonesia dewan pimpinan pusat dpp...,Valid


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("--- Memulai Proses Pembagian Data & Ekstraksi Fitur (TF-IDF) ---")

# 1. Menentukan Fitur (X) dan Target/Label (y)
X = df['Teks Bersih']
y = df['Label']

# 2. Data Splitting: Membagi data latih (80%) dan data uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Jumlah data latih (Training): {len(X_train)} baris")
print(f"Jumlah data uji (Testing): {len(X_test)} baris")

# 3. Feature Extraction (TF-IDF)
print("\nSedang mengubah teks menjadi vektor angka (TF-IDF)...")
# Kita batasi 5000 kata paling penting (max_features) agar komputer Anda tidak kewalahan memproses 31 ribu berita
vectorizer = TfidfVectorizer(max_features=5000) 

# Mempelajari kosakata dari data latih dan mengubahnya ke angka
X_train_tfidf = vectorizer.fit_transform(X_train)
# Mengubah data uji ke angka berdasarkan kosakata data latih
X_test_tfidf = vectorizer.transform(X_test)

print("Proses TF-IDF selesai!")
print(f"Bentuk matriks data latih: {X_train_tfidf.shape}")

--- Memulai Proses Pembagian Data & Ekstraksi Fitur (TF-IDF) ---
Jumlah data latih (Training): 25061 baris
Jumlah data uji (Testing): 6266 baris

Sedang mengubah teks menjadi vektor angka (TF-IDF)...
Proses TF-IDF selesai!
Bentuk matriks data latih: (25061, 5000)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("--- Memulai Pelatihan Model Machine Learning ---")

# 1. Mempersiapkan Algoritma (Sesuai dengan proposal UTS)
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
nb_model = MultinomialNB()

# 2. Melatih Model (Training)
print("Sedang mengajari model Logistic Regression...")
logreg_model.fit(X_train_tfidf, y_train)

print("Sedang mengajari model Naive Bayes...")
nb_model.fit(X_train_tfidf, y_train)

print("\n--- Selesai Belajar! Memulai Evaluasi Ujian (Testing) ---")

# 3. Menguji model dengan Data Testing (X_test_tfidf) yang belum pernah dilihat sebelumnya
prediksi_logreg = logreg_model.predict(X_test_tfidf)
prediksi_nb = nb_model.predict(X_test_tfidf)

# 4. Menampilkan Nilai Rapor (Metrik Evaluasi)
print("\n==============================================")
print("     HASIL EVALUASI LOGISTIC REGRESSION       ")
print("==============================================")
print(f"Akurasi Utama : {accuracy_score(y_test, prediksi_logreg) * 100:.2f}%\n")
print("Metrik Pendukung (Precision, Recall, F1-Score):")
print(classification_report(y_test, prediksi_logreg))

print("\n==============================================")
print("         HASIL EVALUASI NAIVE BAYES           ")
print("==============================================")
print(f"Akurasi Utama : {accuracy_score(y_test, prediksi_nb) * 100:.2f}%\n")
print("Metrik Pendukung (Precision, Recall, F1-Score):")
print(classification_report(y_test, prediksi_nb))

--- Memulai Pelatihan Model Machine Learning ---
Sedang mengajari model Logistic Regression...
Sedang mengajari model Naive Bayes...

--- Selesai Belajar! Memulai Evaluasi Ujian (Testing) ---

     HASIL EVALUASI LOGISTIC REGRESSION       
Akurasi Utama : 99.35%

Metrik Pendukung (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

       Hoaks       0.99      0.99      0.99      2076
       Valid       0.99      1.00      1.00      4190

    accuracy                           0.99      6266
   macro avg       0.99      0.99      0.99      6266
weighted avg       0.99      0.99      0.99      6266


         HASIL EVALUASI NAIVE BAYES           
Akurasi Utama : 98.12%

Metrik Pendukung (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

       Hoaks       0.97      0.98      0.97      2076
       Valid       0.99      0.98      0.99      4190

    accuracy                           0.98      6266
   macro avg       0.98

In [6]:
from sklearn.model_selection import cross_val_score

print("--- Melakukan 5-Fold Cross Validation (Rute Akademis) ---")

# Melakukan ujian silang (cross-validation) 5 kali pada model Logistic Regression
scores = cross_val_score(logreg_model, X_train_tfidf, y_train, cv=5, scoring='accuracy')

print(f"Nilai ujian di masing-masing bagian (5 lipatan):")
for i, skor in enumerate(scores):
    print(f"Ujian ke-{i+1}: {skor * 100:.2f}%")

print(f"\nRata-rata Akurasi Validasi: {scores.mean() * 100:.2f}%")
print("Kesimpulan: Jika rata-rata ini tetap tinggi (di atas 90%), berarti model Anda terbukti stabil secara akademis!")

--- Melakukan 5-Fold Cross Validation (Rute Akademis) ---
Nilai ujian di masing-masing bagian (5 lipatan):
Ujian ke-1: 99.60%
Ujian ke-2: 99.22%
Ujian ke-3: 99.32%
Ujian ke-4: 99.50%
Ujian ke-5: 99.50%

Rata-rata Akurasi Validasi: 99.43%
Kesimpulan: Jika rata-rata ini tetap tinggi (di atas 90%), berarti model Anda terbukti stabil secara akademis!


In [7]:
import joblib

print("--- Menyimpan Otak Model ke dalam File ---")

# 1. Menyimpan kamus kata (TF-IDF Vectorizer) ke dalam folder models
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("✅ File 'tfidf_vectorizer.pkl' berhasil disimpan di folder models!")

# 2. Menyimpan algoritma Logistic Regression ke dalam folder models
joblib.dump(logreg_model, '../models/logreg_model.pkl')
print("✅ File 'logreg_model.pkl' berhasil disimpan di folder models!")

print("\nPersiapan Rute 2 selesai! Cek folder 'models' di VS Code Anda, pasti ada 2 file baru tersebut.")

--- Menyimpan Otak Model ke dalam File ---
✅ File 'tfidf_vectorizer.pkl' berhasil disimpan di folder models!
✅ File 'logreg_model.pkl' berhasil disimpan di folder models!

Persiapan Rute 2 selesai! Cek folder 'models' di VS Code Anda, pasti ada 2 file baru tersebut.
